In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths
from analysis.src.plotting import configure_plotting
from analysis.src.constants import COMPONENTS, COVARIATES, OUTCOMES, RANDOM_STATE
from paths import paths
from preprocessing.src.patients import load_patients

In [ ]:
configure_plotting()

In [ ]:
OUT = experiment_paths('robustness/qof_allocation')

YEARS = (2021, 2022, 2023, 2024, 2025, 2026)
INDICATORS = dict(
    zip(
        ('AF001', 'CHD001', 'DM017', 'HF001', 'HYP001', 'OB002', 'PAD001', 'STIA001'),
        ('AF', 'CHD', 'DM', 'HF', 'HYP', 'OB', 'PAD', 'STIA'),
    )
)

N_SIMULATIONS = 100
NOISE_LEVEL = 0.20

In [ ]:
@dataclass
class AllocationYear:
    year: int
    practice: np.ndarray
    area: np.ndarray
    counts: np.ndarray
    cases: np.ndarray
    denominator: np.ndarray
    lsoas: pd.Index
    target: pd.Index

    def shares(self, multiplier = None):
        weighted = self.counts if multiplier is None else self.counts * multiplier
        totals = np.bincount(self.practice, weights = weighted, minlength = len(self.cases))
        shares = weighted / totals[self.practice]

        return shares

    def allocate(self, shares):
        values = np.zeros_like(self.denominator)

        for j in range(len(OUTCOMES)):
            allocated = self.cases[self.practice, j] * shares
            numerator = np.bincount(self.area, weights = allocated, minlength = len(self.lsoas))
            np.divide(numerator, self.denominator[:, j], out = values[:, j], where = self.denominator[:, j] > 0)

        result = pd.DataFrame(values, index = self.lsoas, columns = OUTCOMES)

        return result.reindex(self.target)

In [ ]:
# Load QOF registers and patient-residence distributions
def load_allocations(primary):
    input_files = [paths.preprocessing.health.qof] + [paths.raw.health.patient_files[y] for y in YEARS]
    missing = [str(p) for p in input_files if not p.is_file()]

    if missing:
        raise FileNotFoundError('Required upstream inputs missing: ' + ', '.join(missing))

    qof = (
        pd.read_csv(input_files[0])
        .rename(columns = {'orgcode_id': 'practice_code'})
        .pivot_table(index = ['practice_code', 'year'], columns = 'areaid_id', values = 'ratio', aggfunc = 'first')
        .rename(columns = INDICATORS)[list(INDICATORS.values())]
    )

    allocations, diagnostics = [], []
    for year in YEARS:
        patients = load_patients(year)
        annual_qof = qof.xs(year - 2000, level = 'year').dropna(how = 'all')
        
        # Retain complete residential support of every QOF practice serving London
        practices = patients.loc[patients.LSOA_CODE.isin(primary.index), 'PRACTICE_CODE'].unique()
        patients = patients[patients.PRACTICE_CODE.isin(annual_qof.index.intersection(practices))]

        p, practice_ids = pd.factorize(patients.PRACTICE_CODE, sort = True)
        a, lsoas = pd.factorize(patients.LSOA_CODE, sort = True)
        n = patients.NUMBER_OF_PATIENTS.to_numpy(dtype = float)

        population = np.bincount(p, weights = n)
        ratios = annual_qof.reindex(practice_ids).to_numpy()
        usable = np.isfinite(ratios)
        cases = np.where(usable, ratios, 0) * population[:, None]

        denominator = np.column_stack(
            [np.bincount(a, weights = n * usable[p, j], minlength = len(lsoas)) for j in range(len(OUTCOMES))]
        )
        denominator.setflags(write = False)

        allocations.append(AllocationYear(year, p, a, n, cases, denominator, pd.Index(lsoas), primary.index))

        outside = ~patients.LSOA_CODE.isin(primary.index).to_numpy()
        diagnostics.append(
            {
                'year': year,
                'practices': len(practice_ids),
                'edges': len(n),
                'outside_london_edges': int(outside.sum()),
                'outside_london_patient_fraction': float(n[outside].sum() / n.sum())
            }
        )

    return allocations, pd.DataFrame(diagnostics)

In [ ]:
# Reconstruct and validate the primary prevalence estimates
def reconstruct(allocations, shares):
    annual = pd.concat({a.year: a.allocate(s) for a, s in zip(allocations, shares)}, names = ['year', 'LSOA21CD'])
    mean = annual.groupby(level = 'LSOA21CD').mean().reindex(allocations[0].target)
    cleaned = mean.mask(mean.gt(mean.quantile(0.999), axis = 1))

    return annual, mean, cleaned

def validate_primary(reconstructed, primary):
    records = []

    for outcome in OUTCOMES:
        expected, actual = primary[outcome], reconstructed[outcome]
        records.append(
            {
                'outcome': outcome,
                'max_absolute_discrepancy': float((actual - expected).abs().max()),
                'missing_mask_differences': int((actual.isna() != expected.isna()).sum())
            }
        )

    return pd.DataFrame(records)

In [ ]:
# Fit the primary association specification
def fit(primary, surface, outcome, sample):
    frame = primary.loc[sample, ['walkability_score'] + COVARIATES].copy()
    frame[outcome] = surface.loc[sample, outcome]

    model = smf.ols(f'{outcome} ~ walkability_score + ' + ' + '.join(COVARIATES), data = frame, missing = 'raise').fit()
    ci = model.conf_int().loc['walkability_score']

    return {
        'coefficient': model.params['walkability_score'],
        'ci_lower': ci.iloc[0],
        'ci_upper': ci.iloc[1],
        'p_value': model.pvalues['walkability_score'],
        'n_lsoas': int(model.nobs)
    }

In [ ]:
tables, figures = OUT.tables, OUT.figures
tables.mkdir(parents = True, exist_ok = True)
figures.mkdir(parents = True, exist_ok = True)

primary = pd.read_parquet(paths.results.slim_15)
enriched = pd.read_parquet(paths.results.enriched_15).reindex(primary.index)
primary[COMPONENTS] = enriched[COMPONENTS]

allocations, coverage = load_allocations(primary)
_, _, baseline = reconstruct(allocations, [a.shares() for a in allocations])

validation = validate_primary(baseline, primary)
validation.to_csv(OUT.qof_allocation_validation_csv, index = False)

display(validation)

In [ ]:
# Simulate uncertainty in practice allocation shares
rng = np.random.default_rng(RANDOM_STATE)

simulations = []
for simulation in range(N_SIMULATIONS):
    shares = []

    for a in allocations:
        multiplier = rng.uniform(1 - NOISE_LEVEL, 1 + NOISE_LEVEL, len(a.counts))
        shares.append(a.shares(multiplier))

    _, _, surface = reconstruct(allocations, shares)

    simulations.append(surface)

    if (simulation + 1) % 10 == 0:
        print(f'Allocated simulation {simulation + 1}/{N_SIMULATIONS}', flush = True)

In [ ]:
# Fit each simulated surface on fixed outcome-specific samples
original_models, mc_rows, sample_rows = {}, [], []
for outcome in OUTCOMES:
    original_sample = complete_case_mask(primary, outcome, COVARIATES)
    sample = original_sample & baseline[outcome].notna()

    for surface in simulations:
        sample &= surface[outcome].notna()

    original_models[outcome] = fit(primary, baseline, outcome, original_sample)
    reference = fit(primary, baseline, outcome, sample)
    b = reference['coefficient']
    
    sample_rows.extend(
        {
            'outcome': outcome,
            'LSOA21CD': area,
            'in_original_sample': bool(original_sample.loc[area]),
            'in_common_sample': bool(sample.loc[area])
        }
        for area in primary.index
    )

    for i, surface in enumerate(simulations):
        mc_rows.append(
            {
                'outcome': outcome,
                'simulation': i + 1,
                'primary_coefficient': b,
                **fit(primary, surface, outcome, sample)
            }
        )

In [ ]:
# Summarise coefficient distributions
full = pd.DataFrame(mc_rows)

summary_rows = []
for outcome, rows in full.groupby('outcome', sort = False):
    coefs = rows.coefficient
    b = rows.primary_coefficient.iloc[0]
    
    summary_rows.append(
        {
            'outcome': outcome,
            'primary_coefficient': b,
            'original_full_sample_coefficient': original_models[outcome]['coefficient'],
            'original_n_lsoas': original_models[outcome]['n_lsoas'],
            'mean': coefs.mean(),
            'median': coefs.median(),
            'sd': coefs.std(ddof = 1),
            'percentile_2_5': coefs.quantile(0.025),
            'percentile_97_5': coefs.quantile(0.975),
            'minimum': coefs.min(),
            'maximum': coefs.max(),
            'same_sign_proportion': (np.sign(coefs) == np.sign(b)).mean(),
            'ci_excludes_zero_proportion': ((rows.ci_lower > 0) | (rows.ci_upper < 0)).mean(),
            'n_simulations': len(rows),
            'n_lsoas': int(rows.n_lsoas.iloc[0])
        }
    )

summary = pd.DataFrame(summary_rows)

display(summary)

In [ ]:
for name, frame in [
    ('mc_summary', summary),
    ('mc_coefficients', full),
    ('samples', pd.DataFrame(sample_rows)),
    ('coverage', coverage)
]:
    frame.to_csv(getattr(OUT, f'qof_allocation_{name}_csv'), index = False)

In [ ]:
# Plot the simulated coefficient distributions
fig, axes = plt.subplots(2, 4, figsize = (14, 7), constrained_layout = True)

for ax, outcome in zip(axes.flat, OUTCOMES):
    rows = full[full.outcome == outcome]
    
    ax.hist(rows.coefficient, bins = 18, color = '#70A6C2', edgecolor = 'white')
    ax.axvline(
        rows.primary_coefficient.iloc[0], color = '#B33B35', linestyle = '--', label = 'Primary residential allocation'
    )
    ax.set(title = outcome.removeprefix('qof_'), xlabel = 'Walkability coefficient', ylabel = 'Simulations')
    ax.ticklabel_format(axis = 'x', style = 'sci', scilimits = (-3, 3))

fig.legend(*axes.flat[0].get_legend_handles_labels(), loc = 'outside upper center', frameon = False)
fig.savefig(OUT.qof_allocation_mc_distributions_png, dpi = 300, bbox_inches = 'tight')

plt.close(fig)